This notebook describes the data preparation pipeline used to convert raw downloaded files into a unified format for systematic evaluation. It consists of two main stages: (1) data extraction and cleaning, and (2) data serialization.

After running this notebook, the cleaned dataset will contain the following files:

- tableA.csv / tableA.jsonl
- tableB.csv / tableB.jsonl
- train.csv
- valid.csv
- test.csv

In [5]:
from pathlib import Path
import zipfile
import gzip
import csv
import json
import shutil

import pandas as pd

# Data Extraction and Cleaning

Specifically, each cleaned dataset should contain the following five files after extraction and clearning:

 - tableA.csv
 - tableB.csv
 - train.csv
 - valid.csv
 - test.csv

The only two exceptions happens on Semi-Heter and Semi-Rel, where tableA and tableB can be in json format.

The purpose of this notebook is therefore to transform heterogeneous data storage structures into the standardized structure described above.


In [4]:
REQUIRED_FILES = [
    "tableA.csv",
    "tableB.csv",
    "train.csv",
    "valid.csv",
    "test.csv",
]


def extract_move_delete(
    raw_zip_path,
    target_dir,
    inner_dir="exp_data",
    required_files=REQUIRED_FILES,
    overwrite=True,
):
    """
    Extract a zip file, move selected files from an inner folder to a target folder,
    and delete the temporary extracted folder.

    Parameters
    ----------
    raw_zip_path : str or Path
        Path to the raw downloaded zip file.

    target_dir : str or Path
        Directory where the cleaned files should be placed.

    inner_dir : str
        Folder inside the zip file that contains the required files.

    required_files : list[str]
        Files expected inside the inner folder.

    overwrite : bool
        If True, overwrite existing files in target_dir.
        If False, raise an error when a target file already exists.
    """
    raw_zip_path = Path(raw_zip_path)
    target_dir = Path(target_dir)

    if not raw_zip_path.exists():
        raise FileNotFoundError(f"Zip file not found: {raw_zip_path}")

    target_dir.mkdir(parents=True, exist_ok=True)

    tmp_extract_dir = raw_zip_path.parent / f"tmp_{raw_zip_path.stem}_extract"

    if tmp_extract_dir.exists():
        shutil.rmtree(tmp_extract_dir)

    tmp_extract_dir.mkdir(parents=True, exist_ok=True)

    try:
        with zipfile.ZipFile(raw_zip_path, "r") as zip_ref:
            zip_ref.extractall(tmp_extract_dir)

        if inner_dir is None:
            source_dir = tmp_extract_dir
        else:
            source_dir = tmp_extract_dir / inner_dir

        if not source_dir.exists():
            raise FileNotFoundError(f"Expected inner folder not found: {source_dir}")

        for filename in required_files:
            src = source_dir / filename
            dst = target_dir / filename

            if not src.exists():
                raise FileNotFoundError(f"Missing required file: {src}")

            if dst.exists() and not overwrite:
                raise FileExistsError(f"Target file already exists: {dst}")

            shutil.move(str(src), str(dst))

        print(f"Finished preparing dataset: {target_dir}")
        print("Files in target folder:")
        for path in sorted(target_dir.iterdir()):
            print(path)

    finally:
        if tmp_extract_dir.exists():
            shutil.rmtree(tmp_extract_dir)


def reorder_columns(
    target_dir,
    files,
    column_order,
    overwrite=True,
):
    target_dir = Path(target_dir)

    for filename in files:
        path = target_dir / filename
        df = pd.read_csv(path)

        missing_cols = [col for col in column_order if col not in df.columns]
        if missing_cols:
            raise ValueError(f"{filename} is missing columns: {missing_cols}")

        df = df[list(column_order)]

        if overwrite:
            output_path = path
        else:
            output_path = path.with_name(f"{path.stem}_reordered.csv")

        df.to_csv(output_path, index=False)
        print(f"Saved: {output_path}")


def rename_columns(target_dir, files, rename_map, overwrite=True):
    target_dir = Path(target_dir)

    for filename in files:
        path = target_dir / filename
        df = pd.read_csv(path)

        missing_cols = [col for col in rename_map if col not in df.columns]
        if missing_cols:
            raise ValueError(f"{filename} is missing columns: {missing_cols}")

        df = df.rename(columns=rename_map)

        output_path = path if overwrite else path.with_name(f"{path.stem}_renamed.csv")
        df.to_csv(output_path, index=False)

        print(f"Saved: {output_path}")


def drop_columns(file_path, columns, overwrite=True):
    file_path = Path(file_path)

    df = pd.read_csv(file_path, encoding="utf-8")

    missing_cols = [col for col in columns if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing columns in {file_path.name}: {missing_cols}")

    df = df.drop(columns=columns)

    output_path = file_path if overwrite else file_path.with_name(
        f"{file_path.stem}_dropped.csv"
    )

    df.to_csv(output_path, index=False, encoding="utf-8")
    print(f"Saved: {output_path}")

## DBLP-Scholar

The five files are already included in the downloaded raw ZIP archive, so we simply extract them and move them to the appropriate target folder.

In [3]:
extract_move_delete(
    raw_zip_path="raw/dblp_scholar_exp_data.zip",
    target_dir="cleaned/dblp_scholar",
    inner_dir="exp_data",
)

Finished preparing dataset: cleaned/dblp_scholar
Files in target folder:
cleaned/dblp_scholar/tableA.csv
cleaned/dblp_scholar/tableB.csv
cleaned/dblp_scholar/test.csv
cleaned/dblp_scholar/train.csv
cleaned/dblp_scholar/valid.csv


## Walmart-Amazon

The five files are already included in the downloaded raw ZIP archive, so we simply extract them and move them to the appropriate target folder.

In [4]:
extract_move_delete(
    raw_zip_path="raw/walmart_amazon_exp_data.zip",
    target_dir="cleaned/walmart_amazon",
    inner_dir="exp_data",
)

Finished preparing dataset: cleaned/walmart_amazon
Files in target folder:
cleaned/walmart_amazon/tableA.csv
cleaned/walmart_amazon/tableB.csv
cleaned/walmart_amazon/test.csv
cleaned/walmart_amazon/train.csv
cleaned/walmart_amazon/valid.csv


## Amazon-Google

The five files are already included in the downloaded raw ZIP archive, so we simply extract them and move them to the appropriate target folder.

In [5]:
extract_move_delete(
    raw_zip_path="raw/amazon_google_exp_data.zip",
    target_dir="cleaned/amazon_google",
    inner_dir=None,
)

Finished preparing dataset: cleaned/amazon_google
Files in target folder:
cleaned/amazon_google/tableA.csv
cleaned/amazon_google/tableB.csv
cleaned/amazon_google/test.csv
cleaned/amazon_google/train.csv
cleaned/amazon_google/valid.csv


## Abt-Buy
The five files are already included in the downloaded raw ZIP archive, so we simply extract them and move them to the appropriate target folder.

To facilitate later use, we reorder the columns as `[name, price, description]`.

In [6]:
extract_move_delete(
    raw_zip_path="raw/abt_buy_exp_data.zip",
    target_dir="cleaned/abt_buy",
    inner_dir="exp_data",
)

Finished preparing dataset: cleaned/abt_buy
Files in target folder:
cleaned/abt_buy/tableA.csv
cleaned/abt_buy/tableB.csv
cleaned/abt_buy/test.csv
cleaned/abt_buy/train.csv
cleaned/abt_buy/valid.csv


In [7]:
reorder_columns(
    target_dir="cleaned/abt_buy",
    files=["tableA.csv", "tableB.csv"],
    column_order=("id", "name", "price", "description"),
)

Saved: cleaned/abt_buy/tableA.csv
Saved: cleaned/abt_buy/tableB.csv


## WDC80

The WDC product series requires additional data-cleaning effort. The data files in the downloaded ZIP archive are organized at the entity-pair level, so we need to convert them into the expected format accordingly.

Here, we clarify several key choices for using WDC:

- WDC80 means that the dataset contains 80% corner cases.
- We use the medium-sized training and validation sets.
- We use the test set that is not observed during the training phase.

Moreover, we rename the `priceCurrency` column to `currency` and reorder the columns to be `[title, brand, price, currency, description]`

In [8]:
def read_json_gz_lines(path):
    rows = []

    with gzip.open(path, "rt", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))

    return pd.DataFrame(rows)


def find_file(root_dir, filename):
    root_dir = Path(root_dir)

    direct_path = root_dir / filename
    if direct_path.exists():
        return direct_path

    matches = list(root_dir.rglob(filename))
    if not matches:
        raise FileNotFoundError(f"Could not find {filename} under {root_dir}")

    if len(matches) > 1:
        print(f"Warning: multiple matches found for {filename}. Using {matches[0]}")

    return matches[0]


def prepare_wdc_product_dataset(
    raw_zip_path,
    target_dir,
    selected_files,
    tmp_dir,
):
    raw_zip_path = Path(raw_zip_path)
    target_dir = Path(target_dir)
    tmp_dir = Path(tmp_dir)

    target_dir.mkdir(parents=True, exist_ok=True)

    if tmp_dir.exists():
        shutil.rmtree(tmp_dir)

    tmp_dir.mkdir(parents=True, exist_ok=True)

    try:
        # -------------------------
        # Extract zip
        # -------------------------
        with zipfile.ZipFile(raw_zip_path, "r") as zip_ref:
            zip_ref.extractall(tmp_dir)

        # -------------------------
        # Read selected pair files
        # -------------------------
        pair_dfs = {}

        for split_name, filename in selected_files.items():
            path = find_file(tmp_dir, filename)
            pair_dfs[split_name] = read_json_gz_lines(path)

            print(f"{split_name}: {filename}")
            print(pair_dfs[split_name].shape)

        # -------------------------
        # Build tableA.csv and tableB.csv
        # -------------------------
        all_pairs = pd.concat(pair_dfs.values(), ignore_index=True)
        
        left_columns = [
            col for col in all_pairs.columns
            if col.endswith("_left") and col != "cluster_id_left"
        ]
        
        right_columns = [
            col for col in all_pairs.columns
            if col.endswith("_right") and col != "cluster_id_right"
        ]
        
        tableA = all_pairs[left_columns].copy()
        tableA.columns = [col.removesuffix("_left") for col in tableA.columns]
        
        tableB = all_pairs[right_columns].copy()
        tableB.columns = [col.removesuffix("_right") for col in tableB.columns]
        
        if "id" not in tableA.columns:
            raise ValueError("tableA does not contain an id column after removing _left suffix")
        
        if "id" not in tableB.columns:
            raise ValueError("tableB does not contain an id column after removing _right suffix")
        
        tableA = tableA.drop_duplicates(subset=["id"]).reset_index(drop=True)
        tableB = tableB.drop_duplicates(subset=["id"]).reset_index(drop=True)
        
        tableA.to_csv(target_dir / "tableA.csv", index=False, encoding="utf-8")
        tableB.to_csv(target_dir / "tableB.csv", index=False, encoding="utf-8")
        
        print("tableA:", tableA.shape)
        print("tableB:", tableB.shape)

        # -------------------------
        # Build train.csv, valid.csv, test.csv
        # -------------------------
        for split_name, df in pair_dfs.items():
            output = df[["id_left", "id_right", "label"]].copy()

            output = output.rename(columns={
                "id_left": "ltable_id",
                "id_right": "rtable_id",
            })

            output = output[["ltable_id", "rtable_id", "label"]]
            output.to_csv(target_dir / f"{split_name}.csv", index=False, encoding="utf-8")

        # -------------------------
        # Final verification
        # -------------------------
        required_files = [
            "tableA.csv",
            "tableB.csv",
            "train.csv",
            "valid.csv",
            "test.csv",
        ]

        for filename in required_files:
            path = target_dir / filename
            if not path.exists():
                raise FileNotFoundError(f"Missing output file: {path}")

        print(f"\nFinished preparing dataset: {target_dir}")
        for path in sorted(target_dir.iterdir()):
            print(path)

    finally:
        if tmp_dir.exists():
            shutil.rmtree(tmp_dir)

In [9]:
prepare_wdc_product_dataset(
    raw_zip_path="raw/80pair.zip",
    target_dir="cleaned/wdc80",
    tmp_dir="raw/tmp_wdcproducts80",
    selected_files={
        "train": "wdcproducts80cc20rnd000un_train_medium.json.gz",
        "valid": "wdcproducts80cc20rnd000un_valid_medium.json.gz",
        "test": "wdcproducts80cc20rnd100un_gs.json.gz",
    },
)

train: wdcproducts80cc20rnd000un_train_medium.json.gz
(6000, 17)
valid: wdcproducts80cc20rnd000un_valid_medium.json.gz
(3500, 17)
test: wdcproducts80cc20rnd100un_gs.json.gz
(4500, 17)
tableA: (3457, 6)
tableB: (3470, 6)

Finished preparing dataset: cleaned/wdc80
cleaned/wdc80/tableA.csv
cleaned/wdc80/tableB.csv
cleaned/wdc80/test.csv
cleaned/wdc80/train.csv
cleaned/wdc80/valid.csv


In [10]:
rename_columns(
    target_dir="cleaned/wdc80",
    files=["tableA.csv", "tableB.csv"],
    rename_map={
        "priceCurrency": "currency",
    },
)

reorder_columns(
    target_dir="cleaned/wdc80",
    files=["tableA.csv", "tableB.csv"],
    column_order=("id", "title", "brand", "price", "currency", "description"),
)

Saved: cleaned/wdc80/tableA.csv
Saved: cleaned/wdc80/tableB.csv
Saved: cleaned/wdc80/tableA.csv
Saved: cleaned/wdc80/tableB.csv


## WDC20

Same as WDC80. The only difference is this is a variant that contains 20\% of corner cases.

In [11]:
prepare_wdc_product_dataset(
    raw_zip_path="raw/20pair.zip",
    target_dir="cleaned/wdc20",
    tmp_dir="raw/tmp_wdcproducts20",
    selected_files={
        "train": "wdcproducts20cc80rnd000un_train_medium.json.gz",
        "valid": "wdcproducts20cc80rnd000un_valid_medium.json.gz",
        "test": "wdcproducts20cc80rnd100un_gs.json.gz",
    },
)

train: wdcproducts20cc80rnd000un_train_medium.json.gz
(6000, 17)
valid: wdcproducts20cc80rnd000un_valid_medium.json.gz
(3500, 17)
test: wdcproducts20cc80rnd100un_gs.json.gz
(4500, 17)
tableA: (3453, 6)
tableB: (3460, 6)

Finished preparing dataset: cleaned/wdc20
cleaned/wdc20/tableA.csv
cleaned/wdc20/tableB.csv
cleaned/wdc20/test.csv
cleaned/wdc20/train.csv
cleaned/wdc20/valid.csv


In [12]:
rename_columns(
    target_dir="cleaned/wdc20",
    files=["tableA.csv", "tableB.csv"],
    rename_map={
        "priceCurrency": "currency",
    },
)

reorder_columns(
    target_dir="cleaned/wdc20",
    files=["tableA.csv", "tableB.csv"],
    column_order=("id", "title", "brand", "price", "currency", "description"),
)

Saved: cleaned/wdc20/tableA.csv
Saved: cleaned/wdc20/tableB.csv
Saved: cleaned/wdc20/tableA.csv
Saved: cleaned/wdc20/tableB.csv


## Rel-Text

The raw archive already contains the five required files. However, tableA is stored as left.txt, while tableB is stored as right.csv. Therefore, we need to convert these files into the expected format. We also add a schema to the ground-truth files so that they are aligned with the other datasets.

In [13]:
raw_dir = Path("raw/rel-text")
target_dir = Path("cleaned/rel-text")
target_dir.mkdir(parents=True, exist_ok=True)

# left.txt -> tableA.csv
with open(raw_dir / "left.txt", "r", encoding="utf-8") as f:
    left_rows = [line.rstrip("\n") for line in f]

tableA = pd.DataFrame({
    "id": range(len(left_rows)),
    "abstract": left_rows,
})

tableA.to_csv(target_dir / "tableA.csv", index=False)

# right.csv -> tableB.csv
shutil.copy2(raw_dir / "right.csv", target_dir / "tableB.csv")

# train/valid/test -> standardized format
for filename in ["train.csv", "valid.csv", "test.csv"]:
    df = pd.read_csv(raw_dir / filename, header=None)

    if df.shape[1] != 3:
        raise ValueError(
            f"{filename} should have exactly 3 columns, but found {df.shape[1]} columns."
        )

    df.columns = ["ltable_id", "rtable_id", "label"]
    df = df[["ltable_id", "rtable_id", "label"]]

    df.to_csv(target_dir / filename, index=False)

print(f"Finished preparing dataset: {target_dir}")
for path in sorted(target_dir.iterdir()):
    print(path)

Finished preparing dataset: cleaned/rel-text
cleaned/rel-text/tableA.csv
cleaned/rel-text/tableB.csv
cleaned/rel-text/test.csv
cleaned/rel-text/train.csv
cleaned/rel-text/valid.csv


## Semi-Heter

The raw folder already contains the five required files. However, both tableA and tableB are stored in JSON format as left.json and right.json. Since the entity records are semi-structured, we keep them in JSONL format instead of converting them to CSV.

The original IDs in the JSON files are inconsistent and noisy. Therefore, we remove the original ID fields and assign new canonical IDs based on row order, starting from 0. We then save the converted files as tableA.jsonl and tableB.jsonl.

Finally, we standardize the ground-truth files by adding the schema [ltable_id, rtable_id, label], so that they are aligned with the other datasets.

In [14]:
raw_dir = Path("raw/semi-heter")
target_dir = Path("cleaned/semi-heter")
target_dir.mkdir(parents=True, exist_ok=True)


# -------------------------
# left.json -> tableA.json
# Use row order as canonical id, starting from 0
# -------------------------
tableA = pd.read_json(raw_dir / "left.json", encoding="utf-8")

id_like_columns = [col for col in tableA.columns if col.lower() == "id"]
if id_like_columns:
    tableA = tableA.drop(columns=id_like_columns)

tableA.insert(0, "id", range(0, len(tableA)))

tableA.to_json(
    target_dir / "tableA.jsonl",
    orient="records",
    lines=True,
    force_ascii=False,
)


# -------------------------
# right.json -> tableB.json
# Use row order as canonical id, starting from 0
# -------------------------
tableB = pd.read_json(raw_dir / "right.json", encoding="utf-8")

id_like_columns = [col for col in tableB.columns if col.lower() == "id"]
if id_like_columns:
    tableB = tableB.drop(columns=id_like_columns)

tableB.insert(0, "id", range(0, len(tableB)))

tableB.to_json(
    target_dir / "tableB.jsonl",
    orient="records",
    lines=True,
    force_ascii=False,
)


# -------------------------
# train/valid/test -> standardized format
# -------------------------
for filename in ["train.csv", "valid.csv", "test.csv"]:
    df = pd.read_csv(raw_dir / filename, header=None, encoding="utf-8")

    if df.shape[1] != 3:
        raise ValueError(
            f"{filename} should have exactly 3 columns, but found {df.shape[1]} columns."
        )

    df.columns = ["ltable_id", "rtable_id", "label"]
    df = df[["ltable_id", "rtable_id", "label"]]

    df.to_csv(target_dir / filename, index=False, encoding="utf-8")


# -------------------------
# Sanity check
# -------------------------
tableA = pd.read_json(target_dir / "tableA.jsonl", lines=True, encoding="utf-8")
tableB = pd.read_json(target_dir / "tableB.jsonl", lines=True, encoding="utf-8")

print("tableA columns:", list(tableA.columns))
print("tableB columns:", list(tableB.columns))

print("tableA id min/max:", tableA["id"].min(), tableA["id"].max())
print("tableB id min/max:", tableB["id"].min(), tableB["id"].max())

print(
    "tableA id follows row order from 0:",
    tableA["id"].equals(pd.Series(range(0, len(tableA)))),
)
print(
    "tableB id follows row order from 0:",
    tableB["id"].equals(pd.Series(range(0, len(tableB)))),
)

tableA columns: ['id', 'title', 'pubyear', 'publisher', 'authors', 'pubmonth', 'pubday', 'edition', 'isbn13', 'language', 'pages', 'series', 'Title', 'Description', 'ISBN', 'ISBN13', 'PageCount', 'FirstAuthor', 'SecondAuthor', 'ThirdAuthor', 'Rating', 'NumberofRatings', 'NumberofReviews', 'Publisher', 'PublishDate', 'Format', 'Language', 'FileName', 'Price', 'Author', 'Publication_Date', 'Dimensions', 'Pages', 'ISBN_13', 'ISBN_10', 'Publisher_dummy', 'Product Dimensions', 'Shipping Weight', 'Edition', 'Paperback', 'Series', 'ASIN', 'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21', 'Sold_by', 'PublicationDate', 'price', 'ProductType']
tableB columns: ['id', 'title', 'pages', 'authors', 'publication_info', 'series', 'language', 'Productdimensions', 'Salesrank', 'ratings', 'prices', 'Pages', 'Dimensions', 'cover', 'price']
tableA id min/max: 0 22132
tableB id min/max: 0 23263
tableA id follows row order from 0: True
tableB id follows row order from 0: True


## Semi-Rel

In this dataset, tableA is stored as left.csv, while tableB is stored as right.json. We keep their formats accordingly and save them as tableA.csv and tableB.json.

We reset the IDs by row order, starting from 0, and standardize the ground-truth files with the schema [ltable_id, rtable_id, label].

In [15]:
raw_dir = Path("raw/semi-rel")
target_dir = Path("cleaned/semi-rel")
target_dir.mkdir(parents=True, exist_ok=True)


# -------------------------
# left.csv -> tableA.csv
# Reset id by row order, starting from 0
# -------------------------
tableA = pd.read_csv(raw_dir / "left.csv", encoding="utf-8")

id_like_columns = [col for col in tableA.columns if col.lower() == "id"]
if id_like_columns:
    tableA = tableA.drop(columns=id_like_columns)

tableA.insert(0, "id", range(len(tableA)))
tableA.to_csv(target_dir / "tableA.csv", index=False, encoding="utf-8")


# -------------------------
# right.json -> tableB.json
# Reset id by row order, starting from 0
# -------------------------
tableB = pd.read_json(raw_dir / "right.json", encoding="utf-8")

id_like_columns = [col for col in tableB.columns if col.lower() == "id"]
if id_like_columns:
    tableB = tableB.drop(columns=id_like_columns)

tableB.insert(0, "id", range(len(tableB)))

tableB.to_json(
    target_dir / "tableB.jsonl",
    orient="records",
    lines=True,
    force_ascii=False,
)


# -------------------------
# train/valid/test -> standardized format
# -------------------------
for filename in ["train.csv", "valid.csv", "test.csv"]:
    df = pd.read_csv(raw_dir / filename, header=None, encoding="utf-8")

    if df.shape[1] != 3:
        raise ValueError(
            f"{filename} should have exactly 3 columns, but found {df.shape[1]} columns."
        )

    df.columns = ["ltable_id", "rtable_id", "label"]
    df = df[["ltable_id", "rtable_id", "label"]]

    df.to_csv(target_dir / filename, index=False, encoding="utf-8")


# -------------------------
# Sanity check
# -------------------------
tableA = pd.read_csv(target_dir / "tableA.csv", encoding="utf-8")
tableB = pd.read_json(target_dir / "tableB.jsonl", lines=True, encoding="utf-8")

print("tableA id min/max:", tableA["id"].min(), tableA["id"].max())
print("tableB id min/max:", tableB["id"].min(), tableB["id"].max())

print(
    "tableA id follows row order from 0:",
    tableA["id"].equals(pd.Series(range(len(tableA)))),
)
print(
    "tableB id follows row order from 0:",
    tableB["id"].equals(pd.Series(range(len(tableB)))),
)

tableA id min/max: 0 29178
tableB id min/max: 0 32822
tableA id follows row order from 0: True
tableB id follows row order from 0: True


# Data Serialization

The serialization step converts each cleaned CSV or JSONL file into a unified two-column CSV file with fields id and text. The output file is saved in the same directory with the `suffix _serialized.csv`.

For CSV files, the id column is used as the record identifier, and all other columns are concatenated into the text field using the format `[column_name] value`. Missing values are replaced with `unknown`.

For JSONL files, each line is parsed as one JSON object. Nested dictionaries are flattened, lists are converted into space-separated strings, and the id field is used as the record identifier. All remaining non-null fields are serialized into the same `[field_name] value` format.

In [6]:
def _flatten(obj):
    """Recursively flatten a nested dict using only the innermost key name."""
    items = {}
    for k, v in obj.items():
        if isinstance(v, dict):
            items.update(_flatten(v))
        elif isinstance(v, list):
            items[k] = " ".join(str(x) for x in v)
        else:
            items[k] = v
    return items


def _serialize_csv(file_path: Path):
    df = pd.read_csv(file_path)
    value_cols = [c for c in df.columns if c != "id"]

    records = []
    for _, row in df.iterrows():
        parts = []
        for col in value_cols:
            val = row[col]
            val = "unknown" if pd.isna(val) else val
            parts.append(f"[{col}] {val}")
        records.append({"id": row["id"], "text": " ".join(parts)})
    return records


def _serialize_jsonl(file_path: Path):
    records = []
    with open(file_path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            flat = _flatten(obj)
            row_id = flat.pop("id")
            parts = [f"[{k}] {v}" for k, v in flat.items() if v is not None]
            records.append({"id": row_id, "text": " ".join(parts)})
    return records


def serialize(file_path: str) -> Path:
    path = Path(file_path)
    suffix = path.suffix.lower()

    if suffix == ".csv":
        records = _serialize_csv(path)
    elif suffix == ".jsonl":
        records = _serialize_jsonl(path)
    else:
        raise ValueError(f"Unsupported file format: {suffix!r}")

    out = path.with_name(path.stem + "_serialized.csv")
    with open(out, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["id", "text"])
        writer.writeheader()
        writer.writerows(records)

    print(f"Wrote {len(records)} records -> {out}")
    return out

In [7]:
data_dir = Path("cleaned")

for table_file in sorted(data_dir.glob("*/table[AB].*")):
    serialize(str(table_file))


Wrote 1081 records -> cleaned/abt_buy/tableA_serialized.csv
Wrote 1092 records -> cleaned/abt_buy/tableB_serialized.csv
Wrote 1363 records -> cleaned/amazon_google/tableA_serialized.csv
Wrote 3226 records -> cleaned/amazon_google/tableB_serialized.csv
Wrote 2616 records -> cleaned/dblp_scholar/tableA_serialized.csv
Wrote 64263 records -> cleaned/dblp_scholar/tableB_serialized.csv
Wrote 2616 records -> cleaned/rel-text/tableA_serialized.csv
Wrote 2294 records -> cleaned/rel-text/tableB_serialized.csv
Wrote 22133 records -> cleaned/semi-heter/tableA_serialized.csv
Wrote 23264 records -> cleaned/semi-heter/tableB_serialized.csv
Wrote 29179 records -> cleaned/semi-rel/tableA_serialized.csv
Wrote 32823 records -> cleaned/semi-rel/tableB_serialized.csv
Wrote 2554 records -> cleaned/walmart_amazon/tableA_serialized.csv
Wrote 22074 records -> cleaned/walmart_amazon/tableB_serialized.csv
Wrote 3453 records -> cleaned/wdc20/tableA_serialized.csv
Wrote 3460 records -> cleaned/wdc20/tableB_seriali

## Extra Verification

In [12]:
data_dir = Path("cleaned")

for dataset_dir in sorted(data_dir.iterdir()):
    if not dataset_dir.is_dir():
        continue

    table_a = pd.read_csv(dataset_dir / "tableA_serialized.csv", index_col="id")
    table_b = pd.read_csv(dataset_dir / "tableB_serialized.csv", index_col="id")

    labels = pd.concat(
        pd.read_csv(f) for f in dataset_dir.glob("*.csv")
        if f.stem not in ("tableA_serialized", "tableB_serialized")
    )

    matches = labels[labels["label"] == 1].head(2)

    print(f"=== {dataset_dir.name} ===")
    for _, row in matches.iterrows():
        print("A:", table_a.loc[row["ltable_id"], "text"], row["ltable_id"])
        print("B:", table_b.loc[row["rtable_id"], "text"], row["rtable_id"])
        print()

=== abt_buy ===
A: [name] belkin cush top for computer laptop f8n044slv [price] unknown [description] belkin cush top for computer laptop f8n044slv uniquely designed soft material for comfortable laptop use on the couch , bed , or floor convenient storage pocket center multi-sides accommodate any side laptop 6 colors to choose from silver finish 588.0
B: [name] belkin cushtop f8n044-slv [price] unknown [description] - silver 231.0

A: [name] sony silver cyber-shot digital camera dscw170 [price] unknown [description] sony silver cyber-shot digital camera dscw170 10.1 megapixels 2.7 ' lcd display 15 mb internal memory carl zeiss 5x optical zoom lens bionz real imaging processor technology 16:9 high-resolution mode 9-point auto focus automatic macro shooting high sensitivity mode burst mode face detection intelligent scene recognition mode in-camera retouching super steadyshot optical image stabilization with anti-blur solution music slideshow hdtv compatibility beginner-friendly function